## Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import defaultdict, Counter
import os
import numpy as np
import math

hf_token = os.environ.get("HF_TOKEN")

plt.style.use('ggplot')
sns.set_palette("viridis")

## Downloading Dataset

In [2]:
print("Downloading dataset from HuggingFace...")

dataset = load_dataset("Bitext/Bitext-customer-support-llm-chatbot-training-dataset", token=hf_token)
print("Download complete")

Download complete


In [3]:
df = dataset['train'].to_pandas()
df.shape

(26872, 5)

In [4]:
df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [5]:
df['word_count'] = df['instruction'].apply(lambda x : len(x.split()))
df.head()

,flags,instruction,category,intent,response,word_count
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,6
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...,9
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,7
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,7
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...,9


In [6]:
corpus = df['instruction'][:1000]
corpus

0       question about cancelling order {{Order Number}}
1      i have a question about cancelling oorder {{Or...
2        i need help cancelling puchase {{Order Number}}
3             I need to cancel purchase {{Order Number}}
4      I cannot afford this order, cancel purchase {{...
                             ...                        
995             how to cancel purchase {{Order Number}}?
996     I want help cancellkng purchase {{Order Number}}
997    I want assistance cancelling purchase {{Order ...
998    I try to change several bloody items of order ...
999                  modifying purchase {{Order Number}}
Name: instruction, Length: 1000, dtype: str

## x  = np.array([[3.0,4.0],[1, 2]])
## np.linalg.norm(x, axis = 1, keepdims=True) is doing this in every row (for x):

np.sqrt(3 ** 2 + 4 ** 2) for row 1 of x which gives 5

np.sqrt(1 ** 2 + 2 ** 2) for row 2 of x which gives 2.23

This vector [5, 2.23] is then the norms variable

All values in x are then divided by this norms variable which should give you np.array([[0.6,0.8],[0.4472136,0.89442719]])

In [7]:
def tokenize(text):
    return text.lower().split()

In [8]:
class CustomIDF:
    
    def __init__(self):
        self.vocab = {}
        self.idf = {}

    def fit_transform(self, corpus):
        N = len(corpus)
        
        # tokenize the corpus
        tokenized_corpus = [tokenize(doc) for doc in corpus]

        # Calculate DF
        DF = defaultdict(int)
        for tokens in tokenized_corpus:
            unique_tokens = set(tokens)
            for token in unique_tokens:
                DF[token] += 1

        # Calculate IDF
        # IDF for a term t is given as IDF(t) = log((1 + N) / (1 + frequency(t)) + 1
        self.vocab = {term:idx for idx, term in enumerate(DF.keys())}
        for term, count in DF.items():
            self.idf[term] = math.log((1 + N) / (1 + count)) + 1

        # Calculate TF
        matrix = np.zeros((N, len(self.vocab)))
        for i, tokens in enumerate(tokenized_corpus):
            total_tokens = len(tokens)
            if total_tokens == 0:
                continue
            term_counts = Counter(tokens)
            for term, count in term_counts.items():
                if term in self.vocab:
                    TF = count / total_tokens
                    j = self.vocab[term]
                    matrix[i][j] = TF * self.idf[term]

        # calculate L2 Norm
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        matrix = np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms!=0)

        return matrix

In [9]:
custom_vectorizer = CustomIDF()
matrix = custom_vectorizer.fit_transform(corpus)
matrix

array([[0.14100015, 0.35033194, 0.26196106, ..., 0.        , 0.        ,
        0.        ],
       [0.08867478, 0.22032322, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.12606189, 0.31321602, 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.18825313, 0.4677377 , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.06113649, 0.        , 0.11358414, ..., 0.43025484, 0.43025484,
        0.        ],
       [0.13578642, 0.        , 0.        , ..., 0.        , 0.        ,
        0.95561197]], shape=(1000, 210))

In [10]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

# ── 1. TF-IDF Vectorization ────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=5000,     # top 5000 terms by TF-IDF score
    stop_words='english', # remove common words
    ngram_range=(1, 2),    # unigrams + bigrams
    min_df=5,              # ignore terms in fewer than 5 docs
    sublinear_tf=True      # log(1 + tf) dampens high-freq terms
)
X_tfidf = tfidf.fit_transform(corpus).toarray()

# ── 2. Fit KMeans ─────────────────────────────────────────────────────
km = KMeans(
    n_clusters=12,
    init='k-means++',  # same as your implementation
    n_init=10,          # 10 restarts, best inertia kept
    max_iter=300,
    tol=1e-4,
    random_state=42
)
labels = km.fit_predict(X_tfidf)

silhouette_score(X_tfidf, labels)

0.2166871782262788

## sklearn's implementation

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

sklearn_vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, norm='l2')
sklearn_tfidf_matrix = sklearn_vectorizer.fit_transform(corpus)
sklearn_tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7406 stored elements and shape (1000, 210)>